In [ ]:
#load data and imports

from sylegendarium import load_experiments
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
df_raw_orig = load_experiments("experiments")
maps = np.load("dataset/dataset_POINTWISE.npz")


In [ ]:
#information about model and initialization
print(df_raw_orig.columns)
results_df = df_raw_orig.copy()
print(results_df['run'].unique())
print(results_df['model_name'].unique())

In [ ]:
#useful functions for plotting and reporting
import ast


def print_example_trajectory_report(results_df, experiment_id):
    example_df = results_df[results_df["run"] == experiment_id]
    rmse = example_df["mse"].values[-1]
    iou = example_df["iou"].values[-1]

    print(
        f"\nExample Trajectory Report for Experiment ID {experiment_id}:\n"
    )
    print(f"RMSE: {rmse:.4f}, IoU: {iou:.4f}")


def plot_example_trajectory(results_df, experiment_id):
    policy_mapping = {
        "epsilon_greedy": r"$\epsilon$-greedy",
        "orienteering": "Receding Horizon\nOrienteering",
        "myopic_greedy": "Value Greedy",
        "mcts": "MCTS",
        "uncertainty_greedy": "Uncertainty Greedy",
    }

    example_df = results_df.loc[
        results_df["run"] == experiment_id
    ].copy()

    example_df["Policy"] = example_df["policy_name"].replace(
        policy_mapping
    )

    policies = list(example_df["Policy"].unique())
    models = list(example_df["model_name"].unique())

    fig, axes = plt.subplots(
        len(policies),
        len(models),
        figsize=(4 * len(models), 4 * len(policies)),
        squeeze=False,
    )

    colors = ["r-", "g-", "b-", "c-"]

    for row, policy in enumerate(policies):
        for col, model in enumerate(models):
            ax = axes[row, col]

            policy_df = example_df[
                (example_df["Policy"] == policy)
                & (example_df["model_name"] == model)
            ]

            if policy_df.empty:
                ax.set_visible(False)
                continue

            trajectories = [[] for _ in range(4)]

            for position_value in policy_df["position"]:
                position = (
                    ast.literal_eval(position_value)
                    if isinstance(position_value, str)
                    else position_value
                )

                for trajectory_id, (y, x) in enumerate(position[:4]):
                    trajectories[trajectory_id].append([x, y])

            ground_truth = policy_df["ground_truth"].values[0]

            ax.imshow(
                ground_truth,
                cmap="viridis",
                interpolation="bicubic",
            )

            for color_id, trajectory in enumerate(trajectories):
                if trajectory:
                    trajectory = np.asarray(trajectory)

                    ax.plot(
                        trajectory[:, 0],
                        trajectory[:, 1],
                        colors[color_id],
                        marker=".",
                        alpha=0.5,
                    )

            ax.set_title(f"{policy}\n{model}")
            ax.set_aspect("equal")
            ax.set_xlabel("X")
            ax.set_ylabel("Y")
            ax.grid(False)

    plt.tight_layout()
    plt.show()

    return fig, axes

def plot_boxplot_models(results_df):
    """
    Plots a boxplot comparing normalized RMSE across models and policies.
    """

    required_columns = {
        "run",
        "step",
        "mse",
        "policy_name",
        "model_name",
    }

    missing_columns = required_columns.difference(results_df.columns)
    if missing_columns:
        raise ValueError(
            f"Faltan columnas requeridas: {sorted(missing_columns)}"
        )

    df = results_df.copy()

    df["mse"] = pd.to_numeric(df["mse"], errors="coerce")

    df = df.sort_values(
        ["run", "policy_name", "model_name", "step"]
    )

    group_columns = ["run", "policy_name", "model_name"]

    valid_mse = df["mse"].where(
        np.isfinite(df["mse"]) & df["mse"].ne(0)
    )

    baseline = (
        valid_mse
        .groupby(
            [df[column] for column in group_columns],
            observed=True,
        )
        .transform("first")
    )

    df["Normalized RMSE"] = df["mse"] / baseline

    df["Normalized RMSE"] = df["Normalized RMSE"].replace(
        [np.inf, -np.inf],
        np.nan,
    )

    df = df.dropna(
        subset=[
            "Normalized RMSE",
            "policy_name",
            "model_name",
        ]
    )

    if df.empty:
        raise ValueError(
            "No hay valores válidos para construir el boxplot. "
            "Comprueba que mse no contenga únicamente NaN o ceros."
        )

    df = df.drop_duplicates(
        subset=group_columns,
        keep="last",
    )

    df = df.rename(
        columns={
            "policy_name": "Policy",
            "model_name": "Model",
            "distance_bin": "Distance (px)",
        }
    )

    policy_mapping = {
        "epsilon_greedy": r"$\epsilon$-greedy",
        "orienteering": "Receding Horizon Orienteering",
        "myopic_greedy": "Value Greedy",
        "mcts": "MCTS",
        "uncertainty_greedy": "Uncertainty Greedy",
    }

    model_mapping = {
        "ensemble": "Deep Ensemble",
        "gaussian_process": "Gaussian Process",
        "edl": "EDL",
        "mcdropout": "MC Dropout",
    }

    df["Policy"] = (
        df["Policy"]
        .astype("string")
        .map(policy_mapping)
        .fillna(df["Policy"].astype("string"))
    )

    df["Model"] = (
        df["Model"]
        .astype("string")
        .map(model_mapping)
        .fillna(df["Model"].astype("string"))
    )

    fig, ax = plt.subplots(figsize=(8, 5))

    sns.boxplot(
        data=df,
        x="Model",
        y="Normalized RMSE",
        hue="Policy",
        palette="tab10",
        showfliers=False,
        ax=ax,
    )

    ax.set_xlabel("Model")
    ax.set_ylabel("Normalized RMSE")
    ax.set_title("Comparison of Normalized RMSE across Models")
    ax.legend(title="Policy")

    fig.tight_layout()
    plt.show()

    return fig, ax

def plot_map_evolution(
    results_df,
    run_id,
    map_column,
    max_columns=5,
    cmap="viridis",
    figsize=None,
):
    """Plot a spatial map at each step of one run."""
    import ast

    required_columns = {"run", "step", map_column, "model_name", "policy_name"}
    missing_columns = required_columns.difference(results_df.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")
    if max_columns < 1:
        raise ValueError("max_columns must be at least 1")

    run_df = results_df.loc[results_df["run"] == run_id].sort_values("step")
    if run_df.empty:
        raise ValueError(f"No rows found for run_id={run_id}")

    maps = []
    steps = []
    identificators = []
    for _, row in run_df.iterrows():
        current_map = row[map_column]
        if isinstance(current_map, str):
            current_map = ast.literal_eval(current_map)
        current_map = np.asarray(current_map)
        if current_map.ndim != 2:
            raise ValueError(
                f"{map_column} at step {row['step']} must be 2D, "
                f"got shape {current_map.shape}"
            )
        maps.append(current_map)
        steps.append(row["step"])
        identificators.append(f"{row['model_name']}_{row['policy_name']}")

    n_panels = len(maps)
    n_columns = min(max_columns, n_panels)
    n_rows = int(np.ceil(n_panels / n_columns))
    if figsize is None:
        figsize = (3.2 * n_columns, 3.2 * n_rows)

    fig, axes = plt.subplots(n_rows, n_columns, figsize=figsize, squeeze=False)
    axes = axes.ravel()
    vmin = min(current_map.min() for current_map in maps)
    vmax = max(current_map.max() for current_map in maps)

    for ax, step, current_map, identificator in zip(axes, steps, maps, identificators):
        image = ax.imshow(current_map, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(f"step {step}_{identificator}")
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

    for ax in axes[n_panels:]:
        ax.remove()

    # fig.suptitle(f"Evolution of {map_column} - run {run_id}")
    fig.tight_layout()
    return fig, axes[:n_panels]


def plot_predicted_mean_evolution(results_df, run_id, **kwargs):
    return plot_map_evolution(
        results_df, run_id, map_column="predicted_mean", **kwargs
    )


def plot_predicted_uncertainty_evolution(results_df, run_id, **kwargs):
    return plot_map_evolution(
        results_df, run_id, map_column="predicted_uncertainty", **kwargs
    )

def plot_obs_map_evolution(results_df, run_id, **kwargs):
    return plot_map_evolution(
        results_df, run_id, map_column="obs_map", **kwargs
    )


# Examples:
# plot_predicted_mean_evolution(results_df, run_id=0)
# plot_predicted_uncertainty_evolution(results_df, run_id=0)
# plt.show()



def heat_map_RMSE_policy_model(results_df):
    """
    Plots a heatmap comparing the RMSE of different policies and models.

    Parameters:
    results_df (pd.DataFrame): A DataFrame containing the results of different policies and models.
    """
    
    df = results_df.copy()

    df["RMSE_normalized"] = df.groupby(["run", "policy_name"])["mse"].transform(lambda x: x / x.iloc[0])  # Normalize RMSE by distance
    
    # Select the last entry for each experiment_id and policy and model to avoid duplicates
    df = df.drop_duplicates(subset=['run', 'policy_name', 'model_name'], keep='last')
    
    df = df.rename(columns={"policy_name": "Policy", "distance_bin": "Distance (px)", "RMSE_normalized": "Normalized RMSE", "model_name": "Model"})

    
    policy_mapping = {
        "epsilon_greedy": r"$\epsilon$-greedy",
        "orienteering": "Receeding Horizon Orienteering",
        "myopic_greedy": "Value Greedy",
        "mcts": "MCTS",
        "uncertainty_greedy": "Uncertainty Greedy",
    }
    df.loc[:, "Policy"] = df["Policy"].replace(policy_mapping)
    
    model_mapping = {
        "ensemble": "Deep Ensemble",
        "gaussian_process": "Gaussian Process",
    }
    df.loc[:, "Model"] = df["Model"].replace(model_mapping)
     
    pivot_table = df.pivot_table(index='Model', columns='Policy', values='Normalized RMSE', aggfunc='mean')
    
    # Order by the average RMSE across policies for better visualization
    pivot_table = pivot_table.loc[pivot_table.mean(axis=1).sort_values().index]
    
    plt.figure(figsize=(8, 6))
    
    sns.heatmap(pivot_table, annot=True, fmt=".2f", cmap="YlGnBu")
    
    plt.title("Heatmap of RMSE by Policy and Model")
    
    plt.tight_layout()
    
    # Show the plot
    # plt.show()

    # plt.savefig(ROOT / "heatmap_rmse_policy_model.png", dpi=300)  # Save the figure with high resolution
    # # Save also as svg for better quality in publications
    # plt.savefig(ROOT / "heatmap_rmse_policy_model.svg")


In [ ]:
#plot trajectories and reports for each experiment
for example_experiment_id in range(len(results_df['run'].unique())):
        plot_example_trajectory(results_df, example_experiment_id)
        print_example_trajectory_report(results_df, example_experiment_id)

In [ ]:
#plot all maps (evolution, mean and uncertainty during a simulation run)
model_df = results_df[results_df["model_name"] == "edl"]

fig, axes = plot_obs_map_evolution(model_df, run_id=0)
fig, axes = plot_predicted_mean_evolution(model_df, run_id=0)
fig, axes = plot_predicted_uncertainty_evolution(model_df, run_id=0)
plt.show()

In [ ]:
#plot heatmap of RMSE for each policy and model
heat_map_RMSE_policy_model(results_df)

In [ ]:
#boxplot comparison of models
plot_boxplot_models(results_df)

In [ ]:
#mean time of each model
def mean_time_per_model(results_df):
    """
    Computes the mean time taken for each model across all runs.

    Parameters:
    results_df (pd.DataFrame): A DataFrame containing the results of different policies and models.

    Returns:
    pd.DataFrame: A DataFrame with the mean time for each model.
    """
    if 'time' not in results_df.columns:
        raise ValueError("The 'time' column is missing from the DataFrame.")

    mean_time_df = results_df.groupby('model_name')['model_mean_time'].mean().reset_index()
    mean_time_df.rename(columns={'model_mean_time': 'Mean Time'}, inplace=True)
    
    return mean_time_df
